In [2]:
# ==============================================================================
# PROJETO: Preparação de Dados e Regressão Linear com PySpark (Módulo 28)
# ==============================================================================

import os, glob
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month
from pyspark.ml.feature import StringIndexer, VectorAssembler, MinMaxScaler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName("PreparacaoDadosRegressaoPySpark") \
    .getOrCreate()

# ------------------------------------------------------------------------------
# 1. Leitura limpa e sem erros da base de vídeos tratados
# ------------------------------------------------------------------------------
print("Arquivos na pasta:", os.listdir("."))

arquivos_parquet = glob.glob("*videos*tratados*.parquet")
arquivos_csv = glob.glob("*videos*tratados*.csv")

if arquivos_parquet:
    df_video = spark.read.parquet(arquivos_parquet[0])
    print(f"Lido com sucesso: {arquivos_parquet[0]}")
elif arquivos_csv:
    df_video = spark.read.option("header", "true").option("inferSchema", "true").csv(arquivos_csv[0])
    print(f"Lido com sucesso via CSV: {arquivos_csv[0]}")
else:
    print("Nenhum arquivo encontrado. Verifique o nome exato na lista acima!")

df_video.printSchema()
df_video.show(5)

Arquivos na pasta: ['.git', '02.ipynb', 'agregacao.ipynb', 'comments.csv', 'join-videos-comments-otimizado', 'join-videos-comments-otimizado.snappy.parquet', 'otimizacao.ipynb', 'preparacao.ipynb', 'tratamento.ipynb', 'USvideos.csv', 'videos-comments-tratados-parquet', 'videos-comments-tratados.snappy.parquet', 'videos-preparados.snappy.parquet', 'videos-stats.csv', 'videos-tratados-parquet']
Lido com sucesso: videos-comments-tratados.snappy.parquet
root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)
 |-- Likes Comment: integer (nullable = true)

+-----------+--------------------+------------+-------+---

In [ ]:
# ------------------------------------------------------------------------------
# 2. Criação das variáveis derivadas e limpeza de nulos
# ------------------------------------------------------------------------------
df_preparado = df_video \
    .withColumn("Month", month(col("Published At"))) \
    .withColumn("Year", year(col("Published At"))) \
    .withColumn(
        "Interaction",
        col("Likes").cast("long") + col("Comments").cast("long") + col("Views").cast("long")
    ) \
    .dropna(subset=["Keyword", "Likes", "Views", "Comments", "Month", "Year", "Interaction"])

In [4]:
# ------------------------------------------------------------------------------
# 3. Divisão em Treino e Teste (80% / 20%) ANTES dos transformadores com fit
# ------------------------------------------------------------------------------
# Evita data leakage ajustando StringIndexer, MinMaxScaler e PCA exclusivamente nos dados de treino
train_df, test_df = df_preparado.randomSplit([0.8, 0.2], seed=42)

In [5]:
# ------------------------------------------------------------------------------
# 4. Indexação da variável categórica 'Keyword' -> 'Keyword Index'
# ------------------------------------------------------------------------------
indexer = StringIndexer(inputCol="Keyword", outputCol="Keyword Index", handleInvalid="keep")
indexer_model = indexer.fit(train_df)

train_df = indexer_model.transform(train_df)
test_df = indexer_model.transform(test_df)

{"ts": "2026-11-18 17:02:15.305", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[ARITHMETIC_OVERFLOW] overflow. Use 'try_add' to tolerate overflow and return NULL instead. If necessary set \"spark.sql.ansi.enabled\" to \"false\" to bypass this error. SQLSTATE: 22003", "context": {"file": "line 8 in cell [4]", "line": "", "fragment": "__add__", "errorClass": "ARITHMETIC_OVERFLOW"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o68.fit.\n: org.apache.spark.SparkArithmeticException: [ARITHMETIC_OVERFLOW] overflow. Use 'try_add' to tolerate overflow and return NULL instead. If necessary set \"spark.sql.ansi.enabled\" to \"false\" to bypass this error. SQLSTATE: 22003\n== DataFrame ==\n\"__add__\" was called from\nline 8 in cell [4]\n\r\n\tat org.apache.spark.sql.errors.ExecutionErrors.arithmeticOverflowError(ExecutionErrors.scala:150)\r\n\tat org.apache.spark.sql.errors.ExecutionErrors.arithmeticOverflowError$(ExecutionErrors.scala:124

ArithmeticException: [ARITHMETIC_OVERFLOW] overflow. Use 'try_add' to tolerate overflow and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22003
== DataFrame ==
"__add__" was called from
line 8 in cell [4]


In [6]:
# ------------------------------------------------------------------------------
# 5. Montagem do vetor com os cinco campos -> 'Features'
# ------------------------------------------------------------------------------
feature_cols = ["Likes", "Views", "Interaction", "Month", "Keyword Index"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="Features")

train_df = assembler.transform(train_df)
test_df = assembler.transform(test_df)


IllegalArgumentException: [FIELD_NOT_FOUND] No such struct field `Keyword Index` in `Video ID`, `Title`, `Published At`, `Keyword`, `Likes`, `Comments`, `Views`, `Interaction`, `Year`, `Comment`, `Sentiment`, `Likes Comment`, `Month`. SQLSTATE: 42704

In [ ]:
# ------------------------------------------------------------------------------
# 6. Normalização do vetor -> 'Features Normal'
# ------------------------------------------------------------------------------
scaler = MinMaxScaler(inputCol="Features", outputCol="Features Normal")
scaler_model = scaler.fit(train_df)

train_df = scaler_model.transform(train_df)
test_df = scaler_model.transform(test_df)

In [ ]:
# ------------------------------------------------------------------------------
# 7. Redução de dimensionalidade com PCA (5 dimensões -> 1 dimensão) -> 'Features PCA'
# ------------------------------------------------------------------------------
pca = PCA(k=1, inputCol="Features Normal", outputCol="Features PCA")
pca_model = pca.fit(train_df)

train_df = pca_model.transform(train_df)
test_df = pca_model.transform(test_df)

train_df.select("Keyword", "Keyword Index", "Features", "Features Normal", "Features PCA").show(5, truncate=False)

In [ ]:
# ------------------------------------------------------------------------------
# 8. Treinamento do Modelo de Regressão Linear para estimar 'Comments'
# ------------------------------------------------------------------------------
lr = LinearRegression(featuresCol="Features Normal", labelCol="Comments", predictionCol="prediction")
lr_model = lr.fit(train_df)

# Aplicamos o modelo nos conjuntos de treino e teste
train_pred = lr_model.transform(train_df)
test_pred = lr_model.transform(test_df)

In [ ]:
# ------------------------------------------------------------------------------
# 9. Avaliação do Modelo (RMSE e R²)
# ------------------------------------------------------------------------------
evaluator_rmse = RegressionEvaluator(labelCol="Comments", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="Comments", predictionCol="prediction", metricName="r2")

rmse_test = evaluator_rmse.evaluate(test_pred)
r2_test = evaluator_r2.evaluate(test_pred)

print(f"RMSE (Teste): {rmse_test:.4f}")
print(f"R²   (Teste): {r2_test:.4f}")

In [ ]:
# ------------------------------------------------------------------------------
# 10. Exportação do DataFrame completo preparado (incluindo os vetores) para 'videos-preparados-parquet'
# ------------------------------------------------------------------------------
# Unimos treino e teste preparados mantendo todas as colunas originais, derivadas e os vetores
# ('Features', 'Features Normal', 'Features PCA') conforme solicitado na atividade.
df_completo_preparado = train_pred.unionByName(test_pred)

print(f"Total de registros no DataFrame completo preparado: {df_completo_preparado.count()}")
df_completo_preparado.printSchema()

try:
    df_completo_preparado.write.mode("overwrite").parquet("videos-preparados-parquet")
    print("DataFrame completo preparado salvo com sucesso em 'videos-preparados-parquet'!")
except Exception as e:
    print("Aviso: HADOOP_HOME/winutils.exe ausente no Windows para gravação nativa Spark.")
    print("Detalhes:", e)